# Enterprise Banking RAG Pipeline

This notebook orchestrates the complete Banking AI Assistant workflow.

User Query
    ↓
Input Security Layer
    ↓
Guardrails Layer
    ↓
Knowledge Base Relevance Check
    ↓
Intent Classification
    ↓
Semantic Search
    ↓
FAISS Retrieval
    ↓
RAG Generation
    ↓
Groundedness Validation
    ↓
Hallucination Detection
    ↓
Multi-LLM Judge
    ↓
Trust Scoring
    ↓
Human Escalation
    ↓
Final Response

### Imports

In [60]:
import os
import re
import json
import pickle
import warnings

import pandas as pd

from scipy.sparse import hstack

from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

warnings.filterwarnings("ignore")

### Load Environment Variables

In [2]:
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

### Load Intent Classification Artifacts

In [3]:
with open(
    "../models/intent_classification/best_intent_classifier.pkl",
    "rb"
) as f:

    intent_model = pickle.load(f)

with open(
    "../models/intent_classification/word_vectorizer.pkl",
    "rb"
) as f:

    word_vectorizer = pickle.load(f)

with open(
    "../models/intent_classification/char_vectorizer.pkl",
    "rb"
) as f:

    char_vectorizer = pickle.load(f)

with open(
    "../models/intent_classification/label_encoder.pkl",
    "rb"
) as f:

    label_encoder = pickle.load(f)

print("Intent Classification Artifacts Loaded")

Intent Classification Artifacts Loaded


### Load Relevance Config

In [4]:
with open(
    "../models/relevance/relevance_config.pkl",
    "rb"
) as f:

    relevance_config = pickle.load(f)

print("Relevance Configuration Loaded")

Relevance Configuration Loaded


### Load Embedding Model

In [5]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding Model Loaded")

Embedding Model Loaded


### Load FAISS Vector Store

In [6]:
vector_db = FAISS.load_local(
    "../vectorstore/faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

print("FAISS Index Loaded")

FAISS Index Loaded


### Load LLM

In [59]:
judge_70b = ChatGroq(
    api_key=GROQ_API_KEY,
    model_name="llama-3.3-70b-versatile",
    temperature=0
)

judge_8b = ChatGroq(
    api_key=GROQ_API_KEY,
    model_name="llama-3.1-8b-instant",
    temperature=0
)

print("LLM Loaded")

LLM Loaded


### Query Preprocessing

In [39]:
def preprocess_text(text):

    text = str(text).lower()

    text = re.sub(
        r'[^a-zA-Z0-9\s]',
        ' ',
        text
    )

    text = re.sub(
        r'\s+',
        ' ',
        text
    ).strip()

    return text

### Intent Prediction

In [40]:
def predict_intent(question):

    clean_question = preprocess_text(
        question
    )

    word_features = (
        word_vectorizer.transform(
            [clean_question]
        )
    )

    char_features = (
        char_vectorizer.transform(
            [clean_question]
        )
    )

    features = hstack([
        word_features,
        char_features
    ])

    prediction = (
        intent_model.predict(features)
    )[0]

    confidence = (
        intent_model.predict_proba(features)
        .max()
        * 100
    )

    label = (
        label_encoder
        .inverse_transform([prediction])[0]
    )

    return {

        "intent": label,
        "confidence": round(confidence,2)

    }

### Banking Knowledge Relevance Check

In [41]:
BANKING_KEYWORDS = [

    "bank",
    "account",
    "loan",
    "emi",
    "deposit",
    "credit",
    "credit card",
    "debit",
    "debit card",
    "upi",
    "neft",
    "rtgs",
    "imps",
    "insurance",
    "fd",
    "rd"
]

def relevance_check(question, keywords=BANKING_KEYWORDS):
    query = question.lower()

    score = 0

    for keyword in keywords:
        if keyword in query:
            score += 10

    return min(score, 100)

### Guardrails Layer

In [42]:
BLOCKED_PATTERNS = [

    "ignore previous instructions",

    "reveal system prompt",

    "show hidden prompt",

    "jailbreak",

    "developer message",

    "bypass security",

    "act as system"

]

def guardrail_check(query):

    query = query.lower()

    for pattern in BLOCKED_PATTERNS:

        if pattern in query:

            return False

    return True

### Semantic Retrieval

In [43]:
def retrieve_documents(
    question,
    k=5
):

    docs = vector_db.similarity_search(
        question,
        k=k
    )

    return docs

### RAG Generation

In [45]:
def generate_answer(
    query,
    context
):

    prompt = f"""

You are a Banking AI Assistant.

Answer ONLY from the context.

If answer not available,
say:

"I could not find that information
in the banking knowledge base."

Context:

{context}

Question:

{query}

"""

    response = judge_70b.invoke(prompt)

    return response.content

### Groundedness Validation

In [49]:
def groundedness_score(
    answer,
    context
):

    prompt = f"""

Rate groundedness between 0 and 100.

Context:

{context}

Answer:

{answer}

Return number only.
"""

    response = judge_70b.invoke(
        prompt
    )

    try:
        return float(
            response.content.strip()
        )

    except:
        return 50

### Hallucination Detection

In [50]:
def hallucination_score(
    answer,
    context
):

    prompt = f"""

Rate hallucination risk.

0 means no hallucination.

100 means severe hallucination.

Context:

{context}

Answer:

{answer}

Return number only.

"""

    response = judge_70b.invoke(
        prompt
    )

    try:
        return float(
            response.content.strip()
        )

    except:
        return 50

### Multi-LLM Judge

In [51]:
def judge_consensus(
    question,
    context,
    answer
):

    prompt = f"""

Question:
{question}

Context:
{context}

Answer:
{answer}

Score answer quality between 0 and 100.

Return number only.
"""

    groq_score_1 = float(
        judge_70b.invoke(prompt)
        .content.strip()
    )

    groq_score_2 = float(
        judge_8b.invoke(prompt)
        .content.strip()
    )

    return round(
        (groq_score_1 + groq_score_2)/2,
        2
    )

### Trust Score

In [52]:
def trust_governance(
    intent_confidence,
    relevance_score,
    groundedness_score,
    hallucination_score,
    consensus_score
):

    score = (

        0.20 * intent_confidence +

        0.25 * relevance_score +

        0.25 * groundedness_score +

        0.15 * (100 - hallucination_score) +

        0.15 * consensus_score

    )

    return round(score, 2)

### Human Escalation

In [53]:
def determine_status(
    trust_score
):

    if trust_score >= 90:

        return "APPROVED"

    elif trust_score >= 75:

        return "CAUTION"

    else:

        return "ESCALATE"

### Enterprise Pipeline

In [54]:
def enterprise_banking_pipeline(
    query
):

    result = {}

    if not guardrail_check(query):

        result["status"] = "BLOCKED"

        result["message"] = (
            "Unsafe query detected."
        )

        return result

    relevance = relevance_check(query)

    if relevance < 10:

        result["status"] = "OUT_OF_SCOPE"

        result["message"] = (
            "Query outside banking domain."
        )

        return result

    intent_output = predict_intent(query)

    intent = intent_output["intent"]
    intent_confidence = intent_output["confidence"]

    context = retrieve_documents(query)

    answer = generate_answer(
        query,
        context
    )

    groundedness = groundedness_score(
        answer,
        context
    )

    hallucination = hallucination_score(
        answer,
        context
    )

    consensus = judge_consensus(
        query,
        context,
        answer
    )

    trust_score = trust_governance(
        intent_confidence,
        relevance,
        groundedness,
        hallucination,
        consensus
    )

    status = determine_status(
        trust_score
    )

    result = {

        "intent":intent,

        "answer":answer,

        "relevance_score":relevance,

        "groundedness_score":groundedness,

        "hallucination_score":hallucination,

        "consensus_score":consensus,

        "trust_score":trust_score,

        "status":status
    }

    return result

### Test Pipeline

In [55]:
query = "How can I reset my internet banking password?"
response = enterprise_banking_pipeline(
    query
)

response

{'intent': 'Digital & Security',
 'answer': "Click 'Forgot Password' on the net banking login page. Verify using your OTP, account number, or debit card details to set a new password.",
 'relevance_score': 20,
 'groundedness_score': 90.0,
 'hallucination_score': 0.0,
 'consensus_score': 92.5,
 'trust_score': 65.44,
 'status': 'ESCALATE'}

### Save Audit Logs

In [56]:
os.makedirs(
    "../logs",
    exist_ok=True
)

with open(
    "../logs/latest_pipeline_run.json",
    "w"
) as f:

    json.dump(
        response,
        f,
        indent=4
    )

### Production Wrapper

In [57]:
def ask_banking_assistant(
    question
):

    return enterprise_banking_pipeline(
        question
    )

### Final Test

In [58]:
ask_banking_assistant(
    "What is UPI?"
)

{'intent': 'Digital & Security',
 'answer': 'UPI is Unified Payments Interface. It lets you send money instantly using just a mobile number, UPI ID, or QR code — no account number needed.',
 'relevance_score': 10,
 'groundedness_score': 90.0,
 'hallucination_score': 0.0,
 'consensus_score': 97.5,
 'trust_score': 71.68,
 'status': 'ESCALATE'}

In [61]:
import sys

print(sys.executable)

f:\PANTA\Projects\Banking-FAQ-RAG-System\banking_ai_assistant_venv\python.exe


## Enterprise Banking RAG Pipeline

This notebook integrates all previously developed modules into a single production workflow.

Pipeline Flow:

User Query
↓
Intent Classification
↓
Knowledge Relevance Check
↓
Guardrails
↓
FAISS Retrieval
↓
RAG Generation
↓
Groundedness Validation
↓
Hallucination Detection
↓
Multi-LLM Consensus
↓
Trust Score Calculation
↓
Human Escalation Decision
↓
Final Trusted Response

Capabilities:

- Intent Detection
- Semantic Retrieval
- RAG-based Answering
- Prompt Injection Protection
- Hallucination Monitoring
- Multi-LLM Validation
- Trust Scoring
- Human Escalation
- Audit Logging

This notebook serves as the master orchestration layer for the Banking Enterprise AI Assistant.